In [47]:
import pandas as pd

In [48]:
products = pd.read_csv('../data/raw/master/products.csv')
customers = pd.read_csv('../data/raw/master/customers.csv')
orders = pd.read_csv('../data/raw/transaction/orders.csv')
order_items = pd.read_csv('../data/raw/transaction/order_items.csv')
payments = pd.read_csv('../data/raw/transaction/payments.csv')
shipments = pd.read_csv('../data/raw/transaction/shipments.csv')
geography = pd.read_csv('../data/raw/master/geography.csv')
returns = pd.read_csv('../data/raw/transaction/returns.csv')
web_traffic = pd.read_csv('../data/raw/analytics-operational/web_traffic.csv')

C:\Users\Admin\AppData\Local\Temp\ipykernel_19336\3921293664.py:4: DtypeWarning: Columns (0: promo_id_2) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv('../data/raw/transaction/order_items.csv')


# **Kiểm tra tính toàn vẹn của dữ liệu của các bảng**

In [49]:
def check_foreign_key(df_child, child_key, df_parent, parent_key, table_name):
    """ 
        Kiểm tra tính toàn vẹn
    """
    mask = df_child[child_key].isin(df_parent[child_key])
    missing_keys = df_child[~mask]
    if missing_keys.empty:
        print(f"Tất cả {child_key} trong {table_name} đều hợp lệ")
    else:
        print(f"Có {len(missing_keys)} dòng trong {table_name} chứa {child_key} không tồn tại ở bảng cha")

In [50]:
check_foreign_key(order_items, 'order_id', orders, 'order_id', 'order_items')
check_foreign_key(order_items, 'product_id', products, 'product_id', 'order_items')
check_foreign_key(orders, 'customer_id', customers, 'customer_id', 'orders')

Tất cả order_id trong order_items đều hợp lệ
Tất cả product_id trong order_items đều hợp lệ
Tất cả customer_id trong orders đều hợp lệ


In [51]:
def check_cardinality(df_base, df_target, key, relation_name):
    """
        Kiểm tra quan hệ 1:1
    """
    key_counts = df_target[key].value_counts()
    violations = key_counts[key_counts > 1]

    if violations.empty:
        print(f"Quan hệ {relation_name} đúng chuẩn 1:1 hoặc 1:0.")
    else:
        print(f"Quan hệ {relation_name} vi phạm! Có {len(violations)}")

In [52]:
check_cardinality(orders, payments, 'order_id', 'orders - payments')
check_cardinality(orders, shipments, 'order_id', 'orders - shipments')

Quan hệ orders - payments đúng chuẩn 1:1 hoặc 1:0.
Quan hệ orders - shipments đúng chuẩn 1:1 hoặc 1:0.


In [53]:
invalid_shipments = shipments.merge(orders[['order_id', 'order_status']], on='order_id', how='left')
invalid_cases = invalid_shipments[~invalid_shipments['order_status'].isin(['shipped', 'delivered', 'returned'])]

In [54]:
if invalid_cases.empty:
    print("Pass")
else:
    print("Not pass")

Pass


# **Data Layering**

In [55]:
df_master = order_items.copy()

df_master = df_master.merge(
    orders[['order_id', 'order_date', 'customer_id', 'order_status']], 
    on = 'order_id', 
    how = 'left'
)

df_master = df_master.merge(
    products[['product_id', 'product_name', 'category', 'segment', 'cogs', 'size']],
    on='product_id', 
    how='left'
)

df_master = df_master.merge(
    customers[['customer_id', 'age_group', 'gender', 'zip']],
    on='customer_id', 
    how='left'
)

df_master = df_master.merge(
    geography[['zip', 'region']],
    on='zip', 
    how='left'
)

In [56]:
print(df_master.shape)
print(df_master.head())

(714669, 19)
   order_id  product_id  quantity  unit_price  discount_amount promo_id  \
0         1        2400         7     1138.22              0.0      NaN   
1         2         609         7    10166.25              0.0      NaN   
2         3         396         3    11220.33              0.0      NaN   
3         4         635         5    10639.25              0.0      NaN   
4         6        1935         1     1597.84              0.0      NaN   

  promo_id_2  order_date  customer_id order_status      product_name  \
0        NaN  2012-07-04        58578    delivered  VietMotion YY-09   
1        NaN  2012-07-04        58621     returned  SaigonFlex UC-74   
2        NaN  2012-07-04        58811    delivered  SaigonFlex UM-01   
3        NaN  2012-07-04        59453    delivered  SaigonFlex UC-00   
4        NaN  2012-07-06        57821    delivered     UrbanVN RP-10   

     category     segment          cogs size age_group  gender   zip region  
0        GenZ      Trendy

# **MCQ**

### **Question 1**

In [57]:
customer_counts = orders['customer_id'].value_counts()
multiorder_customers = customer_counts[customer_counts > 1].index
df_multi = orders[orders['customer_id'].isin(multiorder_customers)].copy()
df_multi = df_multi.sort_values(['customer_id', 'order_date'])
df_multi['order_date'] = pd.to_datetime(df_multi['order_date'])
df_multi['gap_days'] = df_multi.groupby('customer_id')['order_date'].diff().dt.days
df_multi

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source,gap_days
4023,5280,2012-07-25,1,15201,delivered,cod,desktop,paid_search,NaN
143252,184922,2014-05-31,1,15201,returned,credit_card,mobile,referral,675.0
238890,308113,2015-07-31,1,15201,delivered,cod,mobile,paid_search,426.0
374571,483190,2017-04-23,1,15201,delivered,cod,mobile,paid_search,632.0
544446,702081,2020-02-24,1,15201,delivered,credit_card,mobile,organic_search,1037.0
...,...,...,...,...,...,...,...,...,...
549990,709267,2020-04-11,157563,59937,delivered,credit_card,mobile,email_campaign,225.0
557438,718824,2020-05-29,157563,59937,delivered,credit_card,mobile,social_media,48.0
559725,721751,2020-06-22,157563,59937,delivered,paypal,tablet,organic_search,24.0
591841,763157,2021-05-27,157563,59937,delivered,credit_card,mobile,social_media,339.0


In [58]:
median_gap = df_multi['gap_days'].median()
print(f"Q1 - Trung vị số ngày giữa hai lần mua: {median_gap} ngày")

Q1 - Trung vị số ngày giữa hai lần mua: 144.0 ngày


### **Question 2**

In [59]:
# Q2
products['gross_margin'] = (products['price'] - products['cogs']) / products['price']
segment_margin = products.groupby('segment')['gross_margin'].mean().sort_values(ascending=False)

segment_margin

segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: gross_margin, dtype: float64

In [60]:
print("Q2 - Phân khúc có biên lợi nhuận cao nhất:")
print(segment_margin.head(1))

Q2 - Phân khúc có biên lợi nhuận cao nhất:
segment
Standard    0.313442
Name: gross_margin, dtype: float64


### **Question 3**

In [61]:
returns_products = returns.merge(products[['product_id', 'category']], on='product_id', how='inner')
streetwear_returns = returns_products[returns_products['category'] == 'Streetwear']
top_reason = streetwear_returns['return_reason'].value_counts().head(1)

In [62]:
print("Q3 - Lý do trả hàng phổ biến nhất của Streetwear:")
print(top_reason)

Q3 - Lý do trả hàng phổ biến nhất của Streetwear:
return_reason
wrong_size    7626
Name: count, dtype: int64


### **Question 4**

In [63]:
lowest_bounce_rate = web_traffic.groupby('traffic_source')['bounce_rate'].mean().sort_values().head(1)
print("Q4 - Nguồn traffic có tỷ lệ thoát trung bình thấp nhất:")
print(lowest_bounce_rate)

Q4 - Nguồn traffic có tỷ lệ thoát trung bình thấp nhất:
traffic_source
email_campaign    0.004458
Name: bounce_rate, dtype: float64


### **Question 5**

In [64]:
total_items = len(order_items)
promo_items = order_items['promo_id'].notna().sum()

In [65]:
promo_percentage = (promo_items / total_items) * 100
print(f"Q5 - Tỷ lệ dòng có khuyến mãi: {promo_percentage:.2f}%")

Q5 - Tỷ lệ dòng có khuyến mãi: 38.66%


### **Question 6**

In [66]:
valid_customers = customers[customers['age_group'].notna()]
customers_per_age = valid_customers.groupby('age_group')['customer_id'].nunique()
cust_orders = orders.merge(valid_customers[['customer_id', 'age_group']], on='customer_id', how='inner')
orders_per_age = cust_orders.groupby('age_group')['order_id'].nunique()

In [67]:
avg_orders_per_cust = (orders_per_age / customers_per_age).sort_values(ascending=False)
print("Q6 - Nhóm tuổi có số đơn hàng trung bình trên khách hàng cao nhất:")
print(avg_orders_per_cust.head(1))

Q6 - Nhóm tuổi có số đơn hàng trung bình trên khách hàng cao nhất:
age_group
55+    5.406851
dtype: float64


### **Question 7**

In [68]:
order_items['item_revenue'] = order_items['quantity'] * order_items['unit_price']
revenue_df = order_items.merge(orders[['order_id', 'zip']], on='order_id', how='inner')
revenue_df = revenue_df.merge(geography[['zip', 'region']], on='zip', how='inner')

In [69]:
region_revenue = revenue_df.groupby('region')['item_revenue'].sum().sort_values(ascending=False)
print("Q7 - Vùng tạo ra tổng doanh thu cao nhất:")  
print(region_revenue.head(1))

Q7 - Vùng tạo ra tổng doanh thu cao nhất:
region
East    7.637533e+09
Name: item_revenue, dtype: float64


### **Question 8**

In [70]:
cancelled_orders = orders[orders['order_status'] == 'cancelled']
top_payment_cancelled = cancelled_orders['payment_method'].value_counts()
print("Q8 - Phương thức thanh toán dùng nhiều nhất cho đơn cancelled:")
print(top_payment_cancelled.head(1))

Q8 - Phương thức thanh toán dùng nhiều nhất cho đơn cancelled:
payment_method
credit_card    28452
Name: count, dtype: int64


### **Question 9**

In [71]:
target_sizes = ['S', 'M', 'L', 'XL']

items_with_size = order_items.merge(products[['product_id', 'size']], on='product_id', how='inner')
items_with_size = items_with_size[items_with_size['size'].isin(target_sizes)]
total_items_by_size = items_with_size.groupby('size').size()

returns_with_size = returns.merge(products[['product_id', 'size']], on='product_id', how='inner')
returns_with_size = returns_with_size[returns_with_size['size'].isin(target_sizes)]
total_returns_by_size = returns_with_size.groupby('size').size()

In [72]:
return_rates = (total_returns_by_size / total_items_by_size).sort_values(ascending=False)
print("Q9 - Kích thước có tỷ lệ trả hàng cao nhất:")
print(return_rates.head(1))

Q9 - Kích thước có tỷ lệ trả hàng cao nhất:
size
S    0.056515
dtype: float64


### **Question 10**

In [73]:
avg_payment_by_installment = payments.groupby('installments')['payment_value'].mean().sort_values(ascending=False)

print("Q10 - Kế hoạch trả góp có giá trị thanh toán trung bình trên đơn hàng cao nhất:")
print(avg_payment_by_installment.head(1))

Q10 - Kế hoạch trả góp có giá trị thanh toán trung bình trên đơn hàng cao nhất:
installments
6    24446.654403
Name: payment_value, dtype: float64
